In [12]:
import pandas as pd

In [13]:
path= "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_FINAL_LIGHT.csv"
data = pd.read_csv(path)

In [14]:
data

,ticket_uuid,attachment_id,invoice_page_start,invoice_page_end,s3_key,s3_bucket,textract_job_id,number_of_pages,source,is_ve_with_invoice,is_invoice_inside,local_file_path,textract_s3_link,cleaned_text
0,21ff29f9-0939-5ba3-b632-b77ea3077483,f0ac7d1c-7e27-5c24-9f8f-7a4e40ed833a,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,ef42e149fb904aafacc40b21da3f74aef2905e52f8271b...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nhelga gerstl\nhauptgerichtsvollziehe...
1,ad7254c5-f694-5f0b-8562-6148f0881523,7def55aa-6fcc-5904-b12f-dc882a2ca729,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81764bc01661bb977efafe2719191ce13aee52f98172ae...,9,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieherin\n56290 mack...
2,6114ea8c-0415-539d-bbb2-43f2d8d5efcd,0c44f470-a74a-51b1-9e4a-fd47bbf492d8,6.0,6.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,75d7deaabda31af5ee5928ffcb38af9d1c3751f7d28132...,6,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nanlage zur niederschrift d.:\nogv ko...
3,8275e126-f6f4-563b-9336-3e4b41dd1522,09d7bb75-16a0-5049-9feb-c6b00372861c,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81afd26f8317b8190a662546f16b303c19a5111ad5f98c...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\nrutesheimer st...
4,fa82e542-c3a6-5eea-b252-a9e4e3e8bed3,347853fc-8b9f-522a-a40d-21bf915b1af9,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,c5e37ad57c30062544f7d687aa8b12c44a542b18802eec...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\npoststraße 1\n...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,c9a67751-117a-511f-a3ba-707465ffccde,60060512,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_349816/600...,pair-data-engineering-new,0770238c1e81a6396322ec541fe565db058334efb7a1fd...,3,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\npatrick dangl\njohn-f.-kennedy-straß...
298,b11e0d18-3d0e-563d-bb7d-3d5f0d394b2a,60062385,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350075/600...,pair-data-engineering-new,304777b40ee39bcbf467dce2b85c7ff328f877b67169ac...,1,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieher\nbeethovenstraße ...
299,49fabc6f-9f0b-5ac0-85fe-57151c9768c2,60062423,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350085/600...,pair-data-engineering-new,aecb51efe65c0eeda6702957181dfd96722de8d9fa065f...,2,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nm. jödicke\nhellersdorfer weg 35\nob...
300,fa8e0a2c-8d31-5b7a-92e4-99c2a4542f8a,60062417,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350083/600...,pair-data-engineering-new,ebb5ffe1dea08bd3fead45384ef8ab55d6ff1a52758ab8...,5,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nanlage zur niederschrift d. obergeri...


In [15]:
import re
def add_pagestartend_tokens(row):
    row['cleaned_text_with_start_end_tokens'] = row['cleaned_text']
    number_of_pages = row['number_of_pages']
    
    # calculate number of pages from all <page_idx> token occurences
    all_page_token = re.findall(r"<page_(\d+)>", row['cleaned_text'])
    all_page_token = [int(idx) for idx in all_page_token]
    max_page_token = max(all_page_token) if all_page_token else 0
    if max_page_token != number_of_pages:
        print(f"Problem with textract, number_of_pages: {number_of_pages} but max_page_token: {max_page_token} for attachment_id: {row['attachment_id']}")
        
    # go with max page tokens
    
    search_tokens = [f"<page_{idx+1}>" for idx in range(max_page_token)]
    
    for token in search_tokens:
        token_page_idx = int(token.replace("<", "").replace(">", "").split("_")[1])
        next_page_idx = token_page_idx + 1
        is_last_page = token_page_idx == max_page_token
        # replace <page_i> with <page_i_start>
        row['cleaned_text_with_start_end_tokens'] = row['cleaned_text_with_start_end_tokens'].replace(token, f"<page_{token_page_idx}_start>")
        
        if not is_last_page:
            # add <page_i_end> before the next <page_{i+1}>
            row['cleaned_text_with_start_end_tokens'] = row['cleaned_text_with_start_end_tokens'].replace(f"<page_{next_page_idx}>", f"<page_{token_page_idx}_end>\n<page_{next_page_idx}>")
        else:
            # if it's the last page, add <page_i_end> at the end of the text
            row['cleaned_text_with_start_end_tokens'] = row['cleaned_text_with_start_end_tokens'] + f"\n<page_{token_page_idx}_end>"
    
    return row



In [16]:
data = data.apply(lambda row: add_pagestartend_tokens(row), axis=1)

Problem with textract, number_of_pages: 10 but max_page_token: 9 for attachment_id: 91c4d2a0-c7d4-5762-acdf-a38ee8bec7f2
Problem with textract, number_of_pages: 2 but max_page_token: 1 for attachment_id: 60090970
Problem with textract, number_of_pages: 6 but max_page_token: 5 for attachment_id: 60059622
Problem with textract, number_of_pages: 5 but max_page_token: 4 for attachment_id: 60059429


In [17]:
def check(row):
    n_pages = row['number_of_pages']
    must_contain = [f"<page_{idx+1}_start>" for idx in range(n_pages)] + [f"<page_{idx+1}_end>" for idx in range(n_pages)]
    for token in must_contain:
        if token not in row['cleaned_text_with_start_end_tokens']:
            print(f"Missing token: {token} in row with id: {row['attachment_id']}")

In [18]:
data.apply(lambda row: check(row), axis=1)

Missing token: <page_10_start> in row with id: 91c4d2a0-c7d4-5762-acdf-a38ee8bec7f2
Missing token: <page_10_end> in row with id: 91c4d2a0-c7d4-5762-acdf-a38ee8bec7f2
Missing token: <page_2_start> in row with id: 60090970
Missing token: <page_2_end> in row with id: 60090970
Missing token: <page_6_start> in row with id: 60059622
Missing token: <page_6_end> in row with id: 60059622
Missing token: <page_5_start> in row with id: 60059429
Missing token: <page_5_end> in row with id: 60059429


0      None
1      None
2      None
3      None
4      None
       ... 
297    None
298    None
299    None
300    None
301    None
Length: 302, dtype: object

In [19]:
a_ids = ['91c4d2a0-c7d4-5762-acdf-a38ee8bec7f2','91c4d2a0-c7d4-5762-acdf-a38ee8bec7f2','60090970','60059622','60059429']
check = data[data['attachment_id'].isin(a_ids)]

In [20]:
check

,ticket_uuid,attachment_id,invoice_page_start,invoice_page_end,s3_key,s3_bucket,textract_job_id,number_of_pages,source,is_ve_with_invoice,is_invoice_inside,local_file_path,textract_s3_link,cleaned_text,cleaned_text_with_start_end_tokens
25,4fdd851e-46a8-5d56-aa42-dbaa786fb942,91c4d2a0-c7d4-5762-acdf-a38ee8bec7f2,1.0,2.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,7c82c26f47ae657059eb7bb9f13fcab1a93aae26060595...,10,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieher\nschloßstr. 4...,<page_1_start>\nobergerichtsvollzieher\nschloß...
115,e847eef3-4ecf-576c-8cd6-2846f12502b9,60090970,1.0,2.0,ocr_source_files/2026-04-19/egvp_id_350458/600...,pair-data-engineering-new,cf7ee49144d247a7ece0465dc435e264b5b8166981ab90...,2,prod,NaN,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\npetra glöggler\ngerichtsvollzieherin...,<page_1_start>\npetra glöggler\ngerichtsvollzi...
188,e9729c7d-2500-5acb-9ab7-6df5130047dd,60059622,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_349733/600...,pair-data-engineering-new,3f487c1c72ec3948eef5ac49b29a3b93356ea2dc0594c6...,6,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nanlage zur niederschrift d. obergeri...,<page_1_start>\nanlage zur niederschrift d. ob...
267,2ae029f9-03bd-5839-b26d-22fc71954bd4,60059429,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_349704/600...,pair-data-engineering-new,d23bf180872289fa6cda45f39a20362a831d727c6282f2...,5,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieher volker müller...,<page_1_start>\nobergerichtsvollzieher volker ...


In [21]:
for idx, row in check.iterrows():
    print(f"idx: {idx}, attachment_id: {row['attachment_id']}, number_of_pages: {row['number_of_pages']}, local: {row['local_file_path']}")

idx: 25, attachment_id: 91c4d2a0-c7d4-5762-acdf-a38ee8bec7f2, number_of_pages: 10, local: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/invoice_detection_dataset/positive/91c4d2a0-c7d4-5762-acdf-a38ee8bec7f2.pdf
idx: 115, attachment_id: 60090970, number_of_pages: 2, local: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/invoice_detection_dataset/positive/60090970.pdf
idx: 188, attachment_id: 60059622, number_of_pages: 6, local: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/invoice_detection_dataset/negative/60059622.pdf
idx: 267, attachment_id: 60059429, number_of_pages: 5, local: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/invoice_detection_dataset/negative/60059429.pdf


In [22]:
# THESE ARE THE DATA THAT TEXTRACT CANT DETECT. SO JUST SKIP THEM

In [23]:
data

,ticket_uuid,attachment_id,invoice_page_start,invoice_page_end,s3_key,s3_bucket,textract_job_id,number_of_pages,source,is_ve_with_invoice,is_invoice_inside,local_file_path,textract_s3_link,cleaned_text,cleaned_text_with_start_end_tokens
0,21ff29f9-0939-5ba3-b632-b77ea3077483,f0ac7d1c-7e27-5c24-9f8f-7a4e40ed833a,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,ef42e149fb904aafacc40b21da3f74aef2905e52f8271b...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nhelga gerstl\nhauptgerichtsvollziehe...,<page_1_start>\nhelga gerstl\nhauptgerichtsvol...
1,ad7254c5-f694-5f0b-8562-6148f0881523,7def55aa-6fcc-5904-b12f-dc882a2ca729,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81764bc01661bb977efafe2719191ce13aee52f98172ae...,9,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nobergerichtsvollzieherin\n56290 mack...,<page_1_start>\nobergerichtsvollzieherin\n5629...
2,6114ea8c-0415-539d-bbb2-43f2d8d5efcd,0c44f470-a74a-51b1-9e4a-fd47bbf492d8,6.0,6.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,75d7deaabda31af5ee5928ffcb38af9d1c3751f7d28132...,6,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nanlage zur niederschrift d.:\nogv ko...,<page_1_start>\nanlage zur niederschrift d.:\n...
3,8275e126-f6f4-563b-9336-3e4b41dd1522,09d7bb75-16a0-5049-9feb-c6b00372861c,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,81afd26f8317b8190a662546f16b303c19a5111ad5f98c...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\nrutesheimer st...,<page_1_start>\ngerichtsvollzieherin\nruteshei...
4,fa82e542-c3a6-5eea-b252-a9e4e3e8bed3,347853fc-8b9f-522a-a40d-21bf915b1af9,1.0,1.0,data/aftercourt/vermögensverzeichnis_with_invo...,pair-email-classification,c5e37ad57c30062544f7d687aa8b12c44a542b18802eec...,8,raw,True,True,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieherin\npoststraße 1\n...,<page_1_start>\ngerichtsvollzieherin\npoststra...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,c9a67751-117a-511f-a3ba-707465ffccde,60060512,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_349816/600...,pair-data-engineering-new,0770238c1e81a6396322ec541fe565db058334efb7a1fd...,3,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\npatrick dangl\njohn-f.-kennedy-straß...,<page_1_start>\npatrick dangl\njohn-f.-kennedy...
298,b11e0d18-3d0e-563d-bb7d-3d5f0d394b2a,60062385,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350075/600...,pair-data-engineering-new,304777b40ee39bcbf467dce2b85c7ff328f877b67169ac...,1,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\ngerichtsvollzieher\nbeethovenstraße ...,<page_1_start>\ngerichtsvollzieher\nbeethovens...
299,49fabc6f-9f0b-5ac0-85fe-57151c9768c2,60062423,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350085/600...,pair-data-engineering-new,aecb51efe65c0eeda6702957181dfd96722de8d9fa065f...,2,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nm. jödicke\nhellersdorfer weg 35\nob...,<page_1_start>\nm. jödicke\nhellersdorfer weg ...
300,fa8e0a2c-8d31-5b7a-92e4-99c2a4542f8a,60062417,NaN,NaN,ocr_source_files/2026-04-17/egvp_id_350083/600...,pair-data-engineering-new,ebb5ffe1dea08bd3fead45384ef8ab55d6ff1a52758ab8...,5,prod,NaN,False,/Users/melih.gorgulu/Desktop/Projects/aftercou...,s3://pair-data-engineering-new/ocr_prepared_ou...,<page_1>\nanlage zur niederschrift d. obergeri...,<page_1_start>\

In [24]:
# save the updated data with start and end tokens back to csv
data.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_FINAL_LIGHT_with_start_end_tokens.csv", index=False)